# 08 · Reentrenamiento Automático — Modelo de Riesgo CFE

Ejecución semanal programada que:
1. Regenera features desde la capa **Gold**
2. Detecta **data drift** (Kolmogorov-Smirnov + PSI)
3. Evalúa el modelo **champion** actual sobre datos frescos
4. **Re-entrena solo si es necesario** y registra nueva versión si supera al champion

In [0]:
import os, mlflow, json
from datetime import datetime

# --- Cargar config.py ---
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_nb_dir = os.path.dirname(_nb_path)
exec(open(f"/Workspace{_nb_dir}/config.py").read())

_user = spark.sql("SELECT current_user()").first()[0]
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{WORK_SCHEMA}`")

SCHEMA = WORK_SCHEMA
MODEL_NAME = f"{CATALOG}.{SCHEMA}.modelo_riesgo_obras"
EXPERIMENT_NAME = f"/Users/{_user}/cfe_risk_model"
TRAINING_TABLE = f"{CATALOG}.{SCHEMA}.training_riesgo_obras"

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)

# --- Umbrales de decisión ---
DRIFT_PSI_THRESHOLD = 0.2       # PSI > 0.2 = drift significativo
DRIFT_KS_PVALUE = 0.01          # p-value < 0.01 = distribución cambio
MIN_F1_IMPROVEMENT = 0.005      # nuevo modelo debe superar champion por al menos 0.5%
MAX_F1_DEGRADATION = 0.02       # si champion baja más de 2% en datos frescos, re-entrenar

print(f"📋 Config: CATALOG={CATALOG}, SCHEMA={SCHEMA}")
print(f"📦 Model: {MODEL_NAME}")
print(f"⏰ Run: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"\n🎯 Umbrales: PSI>{DRIFT_PSI_THRESHOLD}, KS p<{DRIFT_KS_PVALUE}, F1 degradation>{MAX_F1_DEGRADATION}")

In [0]:
from pyspark.sql import functions as F, Window

# --- Leer de capa GOLD (misma lógica que lab 07) ---
fact = spark.table(f"{CATALOG}.{SCHEMA}.gold_fact_obras")
dim_proy = spark.table(f"{CATALOG}.{SCHEMA}.gold_dim_proyectos")
dim_tipo = spark.table(f"{CATALOG}.{SCHEMA}.gold_dim_tipos_obra")
dim_res  = spark.table(f"{CATALOG}.{SCHEMA}.gold_dim_residencias")

df = (
    fact
    .join(dim_proy.select("IdProyecto", "Cliente", "Contratista"), on="IdProyecto", how="left")
    .join(dim_tipo, on="IdTipoObra", how="left")
    .join(dim_res.select(F.col("IdResidenciaRegional"), F.col("Nombre").alias("Residencia")),
          on="IdResidenciaRegional", how="left")
)

w_proj = Window.partitionBy("IdProyecto")

df_feat = df.select(
    F.col("IdObra"), F.col("NombreObra"), F.col("IdProyecto"),
    F.when((F.col("ImporteContratoMxn") > 0), F.col("ICM") / F.col("ImporteContratoMxn")).otherwise(0).alias("ratio_icm_contrato"),
    F.when((F.col("ImporteContratoMxn") > 0), F.col("CPTT") / F.col("ImporteContratoMxn")).otherwise(0).alias("ratio_cptt_contrato"),
    F.log1p(F.coalesce(F.col("ImporteContratoMxn"), F.lit(0))).alias("log_importe"),
    F.coalesce(F.col("AvanceActualConstruccion"), F.lit(0)).alias("avance_construccion"),
    F.coalesce(F.col("AvanceGeneralAP"), F.lit(0)).alias("avance_ap"),
    (F.coalesce(F.col("AvanceActualConstruccion"), F.lit(0)) - F.coalesce(F.col("AvanceGeneralAP"), F.lit(0))).alias("gap_avance"),
    F.datediff(F.current_date(), F.coalesce(F.col("FechaInicioReal"), F.col("FechaInicioProgramada"))).alias("dias_transcurridos"),
    F.datediff(F.col("FechaTerminacionProgramada"), F.col("FechaInicioProgramada")).alias("dias_programados"),
    F.col("TipoObra").alias("tipo_obra"), F.col("kV").alias("voltaje_kv"), F.col("Residencia").alias("residencia"),
    F.coalesce(F.col("Kilometros"), F.lit(0.0)).alias("kilometros"),
    F.when(F.lower(F.col("SituacionActual")).contains("suspendida"), 1).otherwise(0).alias("esta_suspendida"),
    F.count("IdObra").over(w_proj).alias("num_obras_proyecto"),
)

df_feat = df_feat.withColumn("ratio_tiempo",
    F.when(F.col("dias_programados") > 0, F.col("dias_transcurridos") / F.col("dias_programados")))

df_feat = df_feat.withColumn("en_riesgo",
    F.when(
        ((F.col("avance_construccion") < 30) & (F.coalesce(F.col("ratio_tiempo"), F.lit(0)) > 0.5)) |
        ((F.col("avance_construccion") < 50) & (F.coalesce(F.col("ratio_tiempo"), F.lit(0)) > 0.8)) |
        (F.col("esta_suspendida") == 1) | (F.col("ratio_icm_contrato") > 1.2), 1
    ).otherwise(0))

df_fresh = df_feat.na.fill(0, subset=["ratio_icm_contrato", "ratio_cptt_contrato",
                                       "dias_transcurridos", "dias_programados", "ratio_tiempo"])
fresh_pdf = df_fresh.toPandas()
print(f"✅ Datos frescos: {len(fresh_pdf)} obras, {fresh_pdf['en_riesgo'].sum()} en riesgo ({fresh_pdf['en_riesgo'].mean()*100:.1f}%)")

In [0]:
import numpy as np
from scipy import stats

# --- Cargar datos con los que se entrenó el champion ---
old_pdf = spark.table(TRAINING_TABLE).toPandas()

numeric_features = ["ratio_icm_contrato", "ratio_cptt_contrato", "log_importe",
                    "avance_construccion", "avance_ap", "gap_avance",
                    "dias_transcurridos", "dias_programados", "ratio_tiempo",
                    "kilometros", "esta_suspendida", "num_obras_proyecto"]

# --- PSI (Population Stability Index) ---
def calculate_psi(expected, actual, bins=10):
    """PSI: <0.1 no change, 0.1-0.2 moderate, >0.2 significant drift."""
    breakpoints = np.percentile(expected.dropna(), np.linspace(0, 100, bins + 1))
    breakpoints = np.unique(breakpoints)
    if len(breakpoints) < 3:
        return 0.0
    expected_counts = np.histogram(expected.dropna(), bins=breakpoints)[0] / len(expected.dropna())
    actual_counts = np.histogram(actual.dropna(), bins=breakpoints)[0] / len(actual.dropna())
    expected_counts = np.clip(expected_counts, 0.001, None)
    actual_counts = np.clip(actual_counts, 0.001, None)
    return np.sum((actual_counts - expected_counts) * np.log(actual_counts / expected_counts))

# --- Ejecutar tests ---
drift_results = []
for feat in numeric_features:
    old_vals = old_pdf[feat].dropna()
    new_vals = fresh_pdf[feat].dropna()
    
    ks_stat, ks_pval = stats.ks_2samp(old_vals, new_vals)
    psi = calculate_psi(old_vals, new_vals)
    
    has_drift = psi > DRIFT_PSI_THRESHOLD or ks_pval < DRIFT_KS_PVALUE
    drift_results.append({
        "feature": feat,
        "psi": round(psi, 4),
        "ks_statistic": round(ks_stat, 4),
        "ks_pvalue": round(ks_pval, 4),
        "drift_detected": has_drift
    })

import pandas as pd
df_drift = pd.DataFrame(drift_results)
drifted = df_drift[df_drift["drift_detected"]]

print(f"\n{'='*70}")
print(f"DATA DRIFT REPORT ({datetime.now().strftime('%Y-%m-%d')})")
print(f"{'='*70}")
print(f"Features analizados: {len(numeric_features)}")
print(f"Features con drift:  {len(drifted)} {'\u26a0\ufe0f DRIFT DETECTADO' if len(drifted) > 0 else '\u2705 SIN DRIFT'}")
display(df_drift.sort_values("psi", ascending=False))

data_drift_detected = len(drifted) > 0

In [0]:
from sklearn.metrics import f1_score, classification_report

# --- Cargar champion actual ---
try:
    champion_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
    champion_loaded = True
except Exception as e:
    print(f"⚠️ No se pudo cargar champion: {e}")
    champion_loaded = False

if champion_loaded:
    all_features = numeric_features + ["tipo_obra", "voltaje_kv", "residencia"]
    X_fresh = fresh_pdf[all_features]
    y_fresh = fresh_pdf["en_riesgo"]
    
    # Score champion en datos frescos
    y_pred_champion = champion_model.predict(X_fresh)
    f1_champion_fresh = f1_score(y_fresh, y_pred_champion)
    
    # Obtener F1 original del champion (del run de entrenamiento)
    from mlflow.tracking import MlflowClient
    client = MlflowClient()
    champion_version = client.get_model_version_by_alias(MODEL_NAME, "champion")
    champion_run = mlflow.get_run(champion_version.run_id)
    f1_champion_original = champion_run.data.metrics.get("test_f1", 1.0)
    
    f1_degradation = f1_champion_original - f1_champion_fresh
    performance_degraded = f1_degradation > MAX_F1_DEGRADATION
    
    print(f"\n{'='*70}")
    print(f"EVALUACIÓN DEL CHAMPION (v{champion_version.version})")
    print(f"{'='*70}")
    print(f"F1 original (entrenamiento):  {f1_champion_original:.4f}")
    print(f"F1 en datos frescos:          {f1_champion_fresh:.4f}")
    print(f"Degradación:                  {f1_degradation:+.4f} {'\u26a0\ufe0f DEGRADADO' if performance_degraded else '\u2705 ESTABLE'}")
    print(f"\n{classification_report(y_fresh, y_pred_champion, target_names=['No Riesgo', 'En Riesgo'])}")
else:
    performance_degraded = True
    f1_champion_fresh = 0

In [0]:
# --- Lógica de decisión ---
reasons = []
if data_drift_detected:
    reasons.append(f"Data drift detectado en {len(drifted)} features")
if performance_degraded:
    reasons.append(f"F1 champion degradado en {f1_degradation:+.4f} (umbral: {MAX_F1_DEGRADATION})")
if not champion_loaded:
    reasons.append("No existe modelo champion registrado")

should_retrain = len(reasons) > 0

print(f"\n{'='*70}")
print(f"DECISIÓN DE REENTRENAMIENTO")
print(f"{'='*70}")
if should_retrain:
    print(f"\n\u26a0\ufe0f  REENTRENAMIENTO NECESARIO")
    for r in reasons:
        print(f"   • {r}")
else:
    print(f"\n\u2705  NO SE REQUIERE REENTRENAMIENTO")
    print(f"   • Sin data drift significativo")
    print(f"   • F1 champion estable ({f1_champion_fresh:.4f})")
    print(f"\n   Próxima evaluación: siguiente ejecución programada")

In [0]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from mlflow.models import infer_signature

if should_retrain:
    print("\n🚀 Iniciando reentrenamiento...")
    
    categorical_features = ["tipo_obra", "voltaje_kv", "residencia"]
    X = fresh_pdf[numeric_features + categorical_features]
    y = fresh_pdf["en_riesgo"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    preprocessor = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scl", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="unknown")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ])

    new_model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1))
    ])

    with mlflow.start_run(run_name=f"retrain_{datetime.now().strftime('%Y%m%d')}") as run:
        new_model.fit(X_train, y_train)
        y_pred_new = new_model.predict(X_test)
        f1_new = f1_score(y_test, y_pred_new)

        mlflow.log_metric("test_f1", f1_new)
        mlflow.log_param("retrain_reason", "; ".join(reasons))
        mlflow.log_param("drift_features", len(drifted) if data_drift_detected else 0)
        mlflow.log_metric("champion_f1_fresh", f1_champion_fresh)

        signature = infer_signature(X_train.head(100), new_model.predict(X_train.head(100)))
        mlflow.sklearn.log_model(new_model, "model", signature=signature, input_example=X_train.head(3))
        new_run_id = run.info.run_id

    # --- Comparar con champion ---
    f1_improvement = f1_new - f1_champion_fresh
    should_promote = f1_new >= f1_champion_fresh - MIN_F1_IMPROVEMENT  # promote if at least as good

    print(f"\n{'='*70}")
    print(f"COMPARACIÓN: NUEVO vs CHAMPION")
    print(f"{'='*70}")
    print(f"F1 Champion (datos frescos): {f1_champion_fresh:.4f}")
    print(f"F1 Nuevo modelo:            {f1_new:.4f}")
    print(f"Diferencia:                 {f1_improvement:+.4f}")

    if should_promote:
        # Registrar y promover
        result = mlflow.register_model(f"runs:/{new_run_id}/model", MODEL_NAME)
        client.set_registered_model_alias(MODEL_NAME, "champion", result.version)
        client.set_model_version_tag(MODEL_NAME, result.version, "retrain_date", datetime.now().isoformat())
        client.set_model_version_tag(MODEL_NAME, result.version, "retrain_reason", "; ".join(reasons))
        print(f"\n\u2705 NUEVO CHAMPION: v{result.version} (F1={f1_new:.4f})")
    else:
        print(f"\n\u274c Nuevo modelo NO supera al champion. Se mantiene versión actual.")
else:
    print("\n\u23ed\ufe0f  Reentrenamiento omitido — modelo champion vigente.")
    f1_new = None

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, DoubleType, IntegerType

# --- Guardar reporte de monitoreo ---
report = {
    "run_date": datetime.now().isoformat(),
    "user": _user,
    "data_drift_detected": bool(data_drift_detected),
    "drift_features_count": int(len(drifted)) if data_drift_detected else 0,
    "champion_f1_original": float(f1_champion_original) if champion_loaded else None,
    "champion_f1_fresh": float(f1_champion_fresh),
    "performance_degraded": bool(performance_degraded),
    "retrained": bool(should_retrain),
    "new_model_f1": float(f1_new) if f1_new is not None else None,
    "total_obras": int(len(fresh_pdf)),
    "obras_en_riesgo": int(fresh_pdf["en_riesgo"].sum()),
}

# Schema explícito (necesario cuando new_model_f1 es None)
log_schema = StructType([
    StructField("run_date", StringType()), StructField("user", StringType()),
    StructField("data_drift_detected", BooleanType()), StructField("drift_features_count", IntegerType()),
    StructField("champion_f1_original", DoubleType()), StructField("champion_f1_fresh", DoubleType()),
    StructField("performance_degraded", BooleanType()), StructField("retrained", BooleanType()),
    StructField("new_model_f1", DoubleType()), StructField("total_obras", IntegerType()),
    StructField("obras_en_riesgo", IntegerType()),
])
report_sdf = spark.createDataFrame([report], schema=log_schema)
report_sdf.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.modelo_monitoreo_log")

# --- Actualizar predicciones con champion vigente ---
champion_current = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
all_features = numeric_features + ["tipo_obra", "voltaje_kv", "residencia"]
preds = champion_current.predict(fresh_pdf[all_features])
try:
    probs = champion_current.predict_proba(fresh_pdf[all_features])[:, 1]
except Exception:
    probs = preds.astype(float)

pred_pdf = pd.DataFrame({
    "IdObra": fresh_pdf["IdObra"],
    "NombreObra": fresh_pdf["NombreObra"],
    "risk_probability": probs,
    "risk_label": preds,
    "scored_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
})
spark.createDataFrame(pred_pdf).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.predicciones_riesgo_obras")

# --- Actualizar training table ---
df_fresh.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(TRAINING_TABLE)

print(f"\n{'='*70}")
print(f"RESUMEN FINAL")
print(f"{'='*70}")
print(f"📅 Fecha: {report['run_date'][:16]}")
print(f"📊 Obras: {report['total_obras']} ({report['obras_en_riesgo']} en riesgo)")
print(f"📉 Drift: {'SÍ' if report['data_drift_detected'] else 'NO'} ({report['drift_features_count']} features)")
print(f"🏆 Champion F1: {report['champion_f1_fresh']:.4f}")
print(f"🔄 Reentrenado: {'SÍ' if report['retrained'] else 'NO'}")
if report['new_model_f1']:
    print(f"🆕 Nuevo F1: {report['new_model_f1']:.4f}")
print(f"\n\u2705 Predicciones y training table actualizadas.")
print(f"\u2705 Reporte guardado en {CATALOG}.{SCHEMA}.modelo_monitoreo_log")